# OpenPlaque — sparse coronary topology audit v0.3

This completes exact 3D 6-connected-component measurements for all **640** original ImageCAS-X training/validation cases. It fills the 458-case gap in the v0.2 quality audit.

**No source masks are modified. No test images are opened. No training, model inference, or expert adjudication is performed.**

Use **Runtime → Run all**. CPU suffices. The first cell mounts Drive; the second configures caches.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Reuse/cache controls — second executable cell
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/OpenPlaque')
PACKAGE = ROOT/'OpenPlaque_Evaluation_Dataset_v0_2'
ARCHIVE = ROOT/'OpenPlaque_Sparse_Topology_Audit_v0_3.zip'
SCRIPT = PACKAGE/'openplaque_sparse_topology_v0_3.py'
MASKS = ROOT/'Cache'/'ImageCAS_X_Pretrained_v1'/'ImageCAS-X_dataset.zip_extracted'/'ImageCAS-X_dataset'/'segmentations'
OUTPUT = PACKAGE/'sparse_topology_results_v0_3'
REUSE_CACHED_AUDITS = True
MAX_AUDIT_CASES = None  # do not change for full 640-case run
print('Dataset:', PACKAGE, 'archive found:', ARCHIVE.is_file())

In [ ]:
# Verify frozen source data and make the uploaded sparse topology script available
import json, zipfile, subprocess, sys, importlib.util
assert PACKAGE.is_dir(), f'Missing v0.2 dataset: {PACKAGE}'
assert MASKS.is_dir(), f'Missing ImageCAS-X masks: {MASKS}'
if not SCRIPT.is_file():
    assert ARCHIVE.is_file(), f'Missing sparse audit archive: {ARCHIVE}'
    with zipfile.ZipFile(ARCHIVE) as z:
        src=z.read('OpenPlaque_Evaluation_Dataset_v0_2/openplaque_sparse_topology_v0_3.py')
        SCRIPT.write_bytes(src)
for name in ['train_480_frozen.txt','internal_dev_80_frozen.txt','historical_val_80_frozen.txt','val.txt','split_integrity.json','audit_results/audit_summary.json']:
    assert (PACKAGE/name).is_file(), f'Missing input: {name}'
missing=[p for p in ('numpy','scipy','nibabel','pandas') if importlib.util.find_spec(p) is None]
if missing:subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
split=json.loads((PACKAGE/'split_integrity.json').read_text())
prev=json.loads((PACKAGE/'audit_results'/'audit_summary.json').read_text())
assert (split['future_training_n'],split['new_internal_development_n'],split['original_validation_n'])==(480,80,80)
assert split['development_ids_sha256']=='ef2bd3bdbe7dc9cbf4639e62d80dceb53108ac836acff64617e0f17b4f29be4f'
assert prev['audited_n']==640 and prev['geometry_pass_n']==640 and prev['errors_n']==0
assert prev['test_reads']==0 and prev['verified_negative_voxels']==0
print('PASS: frozen splits, v0.2 audit, and input paths')

In [ ]:
# Run exact sparse 6-neighbor connected-component audit, resuming from cached cases
cmd=[sys.executable,str(SCRIPT),'--dataset',str(PACKAGE),'--mask-dir',str(MASKS),'--output-dir',str(OUTPUT)]
if MAX_AUDIT_CASES is not None:cmd.extend(['--max-cases',str(MAX_AUDIT_CASES)])
if not REUSE_CACHED_AUDITS:cmd.append('--force')
subprocess.run(cmd,check=True)

In [ ]:
# Summarize results and freeze a positive-reference audit manifest if all 640 pass
import pandas as pd,hashlib
from IPython.display import display
summary=json.loads((OUTPUT/'sparse_topology_summary.json').read_text())
results=pd.read_csv(OUTPUT/'per_case_sparse_topology.csv')
print('Processed:',summary['audited_cases'],'of 640. Errors:',summary['errors_n'],'Flagged:',summary['review_cases_n'])
cols=['case_id','role','all_positive_components','all_positive_largest_component_fraction','lm_components','lad_components','lcx_components','rca_components']
display(results[cols].head(12))
if summary['full_audit_complete'] and len(results)==640:
    manifest={'version':'OpenPlaque-positive-reference-QA-v0.3','status':'FROZEN_POSITIVE_REFERENCE_QA_ONLY',
      'audited_n':640,'frozen_dev_sha256':summary['frozen_dev_sha256'],
      'topology_algorithm':summary['version'],
      'topology_csv_sha256':hashlib.sha256((OUTPUT/'per_case_sparse_topology.csv').read_bytes()).hexdigest(),
      'source_masks_modified':False,'sealed_test_accessed':False,
      'verified_negative_voxels':0,'expert_adjudication':False,
      'full_coronary_anatomy_verified':False,
      'next_step':'Train and benchmark new coronary models on frozen 480/80; original validation 80 for historical diagnostics only'}
    dest=OUTPUT/'frozen_positive_reference_manifest_v0_3.json'
    dest.write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n')
    print('Frozen positive-reference QA manifest:',dest)
else: print('HOLD: run full 640-case audit before freezing')
print('Saved outputs:',OUTPUT)